In [33]:
from astropy.io import fits
import numpy as np

from tqdm import tqdm
import os

from glob import glob

In [42]:
synth_dir = './output_full_run_20k/'
blank_euclid_dir = '../euclid/euclid_blank_cutouts_5000_v1/'

synth_file_names = glob(synth_dir + '*.fits')
blank_euclid_file_names = glob(blank_euclid_dir+'VIS/' + '*.fits')

len(synth_file_names), len(blank_euclid_file_names)*4

(19892, 19956)

In [57]:
blank_ids = []

for i, _ in enumerate(blank_euclid_file_names):
    blank_ids.append([i, 0])
    blank_ids.append([i, 1])
    blank_ids.append([i, 2])
    blank_ids.append([i, 3])

blank_ids = np.array(blank_ids)

In [56]:
rng = np.random.default_rng()

rng.shuffle(blank_ids)

blank_ids.shape

(19956, 2)

In [63]:
filters = ['VIS', 'NIR_Y', 'NIR_J', 'NIR_H']

output_dir = synth_dir+'with_blank_euclid_rotation_test_v3/'

os.makedirs(output_dir, exist_ok=True)

broken_files = []

n_to_process = len(synth_file_names)

for idx, synth_file in tqdm(enumerate(synth_file_names[:n_to_process]), total=n_to_process):
    synth_data = {}
    synth_header = {}
    blank_data = {}
    
    try:
        with fits.open(synth_file, ignore_missing_simple=True) as synth_hdul:
            for ext_i, filter in enumerate(filters):
                synth_data[filter] = synth_hdul[ext_i + 1].data
                synth_header[filter] = synth_hdul[ext_i + 1].header

    except Exception as e:
        print(f"Error processing {synth_file}: {e}")
        broken_files.append(synth_file)
        continue
    
    blank_id = blank_ids[idx]
    blank_file = blank_euclid_file_names[blank_id[0]]
    rotation = int(blank_id[1])

    for filter in filters:
        blank_file_name = blank_file.split('/')[-1].removeprefix('VIS')
        with fits.open(blank_euclid_dir + filter + '/' + filter + blank_file_name) as blank_hdul:
            blank_data[filter] = blank_hdul[0].data

        # Rotate the full blank imag    e by 90 degrees * k before cutout.
        blank_data[filter] = np.rot90(blank_data[filter], k=rotation)

    # insert the synth image into the blank euclid image
    for filter in filters:
        synth_h, synth_w = synth_data[filter].shape
        blank_h, blank_w = blank_data[filter].shape

        y0 = blank_h // 2 - synth_h // 2
        y1 = y0 + synth_h
        x0 = blank_w // 2 - synth_w // 2
        x1 = x0 + synth_w

        blank_cutout = blank_data[filter][y0:y1, x0:x1]
        blank_nan_mask = np.isnan(blank_cutout)

        # Zero synth pixels where blank is NaN, then add only finite blank values.
        synth_masked = np.where(blank_nan_mask, 0, synth_data[filter])
        blank_filled = np.where(blank_nan_mask, 0, blank_cutout)
        if filter.startswith('NIR'):
            synth_data[filter] = 0.2*synth_masked + blank_filled
        else:
            synth_data[filter] = synth_masked + blank_filled
        
    # save the new image as a fits file
    new_hdul = fits.HDUList()
    new_hdul.append(fits.PrimaryHDU())
    for filter in filters:
        new_hdul.append(fits.ImageHDU(data=synth_data[filter], header=synth_header[filter], name=filter))
    new_hdul.writeto(output_dir+synth_file.split('/')[-1], overwrite=True)

100%|██████████| 19892/19892 [16:03<00:00, 20.64it/s]


In [26]:
len(glob('./output_larger_test_images_5000/with_blank_euclid/*.fits'))

4966